In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Mini-projet 2 – Effacement de consommation

## Partie 1 : Étude du problème d'optimisation

### Justification de la facture (5)

La puissance $P_i$ est supposée constante sur l'intervalle $[t_i, t_{i+1}]$ de durée $\Delta t$. L'énergie consommée sur cet intervalle est donc $P_i \Delta t$ (en Wh). Le coût associé, au tarif $c_i$, est $c_i P_i \Delta t$. En sommant sur tous les intervalles $i = 0, \ldots, N$, avec la convention $P_N = 0$ (condition terminale (3)), on obtient :
$$\text{Facture} = \Delta t \sum_{i=0}^{N} c_i P_i = \Delta t \sum_{i=0}^{N-1} c_i P_i$$
ce qui est bien la formule (5).

### Interprétation de l'équation (2)

L'équation
$$T_{i+1} = e^{-(k+h)\Delta t} T_i + \frac{1 - e^{-(k+h)\Delta t}}{k+h}\left(bP_i + hT^e_i\right)$$
est la solution exacte discrétisée de l'équation différentielle ordinaire
$$\dot{T}(t) = -(k+h)T(t) + bP(t) + hT^e(t)$$
sur l'intervalle $[t_i, t_{i+1}]$, avec $P$ et $T^e$ constants sur cet intervalle.

Cette EDO modélise **deux échanges thermiques** :
- **Pertes thermiques vers l'extérieur** : le terme $-kT$ modélise les déperditions de chaleur à travers les murs et le toit, proportionnelles à la température intérieure (par rapport à une référence nulle). 
- **Échange convectif avec l'air extérieur** : le terme $h(T^e - T)$ modélise l'échange thermique avec l'air extérieur, proportionnel à l'écart de température $T^e - T$. Il peut être positif (l'extérieur réchauffe) ou négatif (l'extérieur refroidit).
- **Apport de chaleur par le chauffage** : le terme $bP$ représente la puissance thermique injectée par les convecteurs.

La modélisation est **raisonnable** pour un bâtiment résidentiel sur un horizon de quelques heures : elle capture l'inertie thermique via le coefficient $e^{-(k+h)\Delta t}$ (la température ne saute pas brusquement), et les deux mécanismes physiques principaux. Elle reste une simplification (un seul nœud thermique, coefficients constants, pas de rayonnement solaire direct), mais elle est classique en génie thermique et suffisante pour le problème de pilotage considéré.

In [ ]:
# Illustration : température extérieure et dynamique libre (sans chauffage)
t0 = 23.0; dt = 0.5; N = 48
h = 0.05; k = 0.01; b = 1/500
T_in = 18.0

t = np.array([(t0 + i * dt) % 24 for i in range(N + 1)])
T_ext = 4 + 8 * np.exp(-(t - 12)**2 / 40)

alpha = np.exp(-(k + h) * dt)
beta  = (1 - alpha) / (k + h)

# Simulation dynamique libre (P=0)
T_libre = np.zeros(N + 1)
T_libre[0] = T_in
for i in range(N):
    T_libre[i+1] = alpha * T_libre[i] + beta * h * T_ext[i]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, T_ext,   'k--', linewidth=1.5, label='Température extérieure $T^e$')
ax.plot(t, T_libre, 'b-o', markersize=3,  label='Température libre $T$ (sans chauffage)')
ax.set_xlabel('Heure'); ax.set_ylabel('Température (°C)')
ax.set_xticks(range(0, 25, 2)); ax.set_xlim(0, 24)
ax.legend(fontsize=9)
ax.set_title('Dynamique libre du bâtiment (P = 0)')
plt.tight_layout(); plt.show()

On observe que sans chauffage, la température intérieure tend vers la température extérieure (environ 4–12°C), largement en dessous de $T_m = 18°C$ pendant les heures d'occupation. Le chauffage est donc indispensable pour satisfaire les contraintes de confort.

### Formulation du problème d'optimisation

**Variables de décision.** On choisit comme variables de décision les puissances de chauffage et les températures à chaque pas de temps :
$$x = (P_0, \ldots, P_{N-1}, T_0, \ldots, T_N) \in \mathbb{R}^{2N+1}$$
soit $n = 2N + 1$ variables. On aurait pu n'garder que les $P_i$ (les $T_i$ étant déterminés par la récurrence (2)), mais il est commode dans la formulation LP de les conserver explicitement comme variables.

**Fonction objectif.**
$$f(x) = \Delta t \sum_{i=0}^{N-1} c_i P_i$$
C'est une fonction **linéaire** en $x$.

**Contraintes d'égalité** $c_{eq}(x) = 0$ :
- Condition initiale : $T_0 - T_{in} = 0$
- Dynamique (2) pour $i = 0, \ldots, N-1$ :
$$T_{i+1} - e^{-(k+h)\Delta t} T_i - \frac{1-e^{-(k+h)\Delta t}}{k+h} b P_i = \frac{1-e^{-(k+h)\Delta t}}{k+h} h T^e_i$$

Ce sont $N+1$ équations **affines** en $(P_i, T_i)$.

**Contraintes d'inégalité** $c_{in}(x) \leq 0$ :
- Bornes sur la puissance : $-P_i \leq 0$ et $P_i - P_M \leq 0$ pour $i = 0, \ldots, N-1$
- Confort thermique : $T_m - T_i \leq 0$ et $T_i - T_M \leq 0$ pour $i \in I_{occ}$

Ce sont toutes des contraintes **affines** en $x$.

### Convexité et famille du problème

Le problème est une **minimisation d'une fonction linéaire** sous **contraintes affines**. C'est donc un **problème de programmation linéaire (LP)**. Il appartient à la famille des problèmes à contraintes affines étudiée à la section 2.2.2.1 du cours (cas des contraintes affines, problème (34)).

Un LP est toujours **convexe** : la fonction objectif est convexe (linéaire), l'ensemble admissible est un polyèdre convexe (intersection de demi-espaces affines). Par le théorème 25 (conditions KKT pour contraintes affines) du cours, tout minimum local est un minimum global. De plus, d'après la proposition 12 de l'appendice C (méthode du simplexe), s'il existe une solution optimale, il en existe une qui est un sommet du polyèdre admissible.

In [ ]:
# Illustration : structure du problème LP
# Variables, contraintes, structure creuse de la matrice A_eq
N_ex = 6  # petit exemple pour la visualisation

n_var = 2 * N_ex + 1         # N_ex puissances + N_ex+1 températures
n_eq  = N_ex + 1             # 1 condition initiale + N_ex équations de dynamique
n_ineq_P = 2 * N_ex          # bornes sur P
print(f"Pour N={N_ex} : {n_var} variables, {n_eq} contraintes égalité, {n_ineq_P} bornes sur P")
print(f"Pour N=48  : {2*48+1} variables, {48+1} contraintes égalité")

# Matrice A_eq (structure creuse)
alpha_ex = np.exp(-(k + h) * dt)
beta_ex  = (1 - alpha_ex) / (k + h)

A_eq = np.zeros((n_eq, n_var))
b_eq = np.zeros(n_eq)

# T_0 = T_in
A_eq[0, N_ex] = 1.0
b_eq[0] = T_in

# Dynamique
T_ext_ex = 4 + 8 * np.exp(-(np.array([(t0 + i*dt) % 24 for i in range(N_ex+1)]) - 12)**2 / 40)
for i in range(N_ex):
    A_eq[i+1, N_ex + i + 1] =  1.0
    A_eq[i+1, N_ex + i]     = -alpha_ex
    A_eq[i+1, i]            = -beta_ex * b
    b_eq[i+1]               =  beta_ex * h * T_ext_ex[i]

fig, ax = plt.subplots(figsize=(8, 3))
ax.spy(A_eq, markersize=8, color='steelblue')
ax.set_title(f'Structure creuse de $A_{{eq}}$ (N={N_ex}) – {n_eq} lignes × {n_var} colonnes')
ax.set_xlabel(f'Variables : $P_0,\ldots,P_{{N-1}}$ | $T_0,\ldots,T_N$')
ax.set_ylabel('Contraintes égalité')
plt.tight_layout(); plt.show()

La structure creuse de $A_{eq}$ illustre que chaque équation de dynamique ne fait intervenir que **trois variables** : $P_i$, $T_i$ et $T_{i+1}$. C'est une structure **bande** typique des problèmes de commande optimale discrétisés, que les solveurs LP comme HiGHS exploitent efficacement.